In [9]:
import cv2
import matplotlib.pyplot as plt
from GoldenRecords import golden
import pandas as pd
import seaborn as sns


In [10]:
wa=pd.read_csv("wannoreport_may_one_pic_plus_chart.csv")

wa.rename(columns={"filenames":"filename"},inplace=True)

wa=wa.merge(pd.DataFrame(golden).rename(columns={'score':'Actual',"one_thing_to_improve":"CoachFeedback"}),on='filename',how='left')


wa['AbsError']=abs(wa['Actual']-wa['score'])


In [11]:
import pandas as pd
import ollama
from pydantic import BaseModel, Field


class CoachJudgeResult(BaseModel):
    score: int = Field(ge=1, le=10)
   

def judge_coach_alignment(
    llm_feedback: str,
    coach_feedback: str,
    model: str = "gemma4:e4b",
    temperature: float = 0.0,
) -> CoachJudgeResult:
    prompt = f"""
You are an expert discus throwing coach and evaluator.

Your job is to compare two pieces of feedback:

1. LLM feedback
2. Real coach feedback

Score how well the LLM feedback aligns with the real coach feedback.

Scoring rubric:
- same_main_points: Does the LLM identify the same major strengths/flaws?


Overall score:
- 9-10: highly aligned
- 7-8: mostly aligned
- 5-6: partially aligned
- 3-4: weak alignment
- 1-2: poor alignment

Important:
- Do not require word-for-word similarity.
- Reward similar coaching points.
- If the LLM says different words but gives the same coaching direction, score it well.
- Penalize made-up flaws

LLM feedback:
{llm_feedback}

Real coach feedback:
{coach_feedback}
"""

    resp = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        format=CoachJudgeResult.model_json_schema(),
        options={"temperature": temperature},
    )

    return CoachJudgeResult.model_validate_json(resp["message"]["content"])

In [4]:
results = []

count=0
for _, row in wa.iterrows():
    
    llm_feedback = str(row["feedback"])
    coach_feedback = str(row["CoachFeedback"])

    judged = judge_coach_alignment(
        llm_feedback=llm_feedback,
        coach_feedback=coach_feedback,
        model="gemma4:e4b",
        temperature=0
    )

    results.append({
        "llm_feedback": llm_feedback,
        "coach_feedback": coach_feedback,
        "overall_score": judged.score
    })
    count+=1
    print(count)

results_df = pd.DataFrame(results)


1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42


In [5]:
pd.concat([results_df,wa],axis=1).\
    groupby(['model','frames','prompt'])['overall_score'].median().\
    sort_values(ascending=False)

model       frames  prompt               
gemma4:e4b  18      \nYou are a brutally     2.0
                    \nYou are a discus co    1.0
                    \nYou are an elite Ol    1.0
llava:7b    18      \nYou are a brutally     1.0
                    \nYou are a discus co    1.0
                    \nYou are an elite Ol    1.0
Name: overall_score, dtype: float64

In [8]:
pd.concat([results_df,wa],axis=1).\
    groupby(['frames','model','prompt','Actual'])['overall_score'].median().\
    sort_values(ascending=False)

frames  model       prompt                 Actual
18      gemma4:e4b  \nYou are a brutally   7         3.0
                                           5         2.5
                    \nYou are an elite Ol  5         2.0
        llava:7b    \nYou are a discus co  5         2.0
        gemma4:e4b  \nYou are a discus co  5         2.0
                                           7         2.0
        llava:7b    \nYou are an elite Ol  5         1.5
                    \nYou are a brutally   5         1.5
        gemma4:e4b  \nYou are a discus co  9         1.0
                    \nYou are an elite Ol  7         1.0
                                           8         1.0
                                           9         1.0
                    \nYou are a discus co  8         1.0
        llava:7b    \nYou are a brutally   7         1.0
                                           8         1.0
                                           9         1.0
        gemma4:e4b  \nYou are a brutal

In [7]:
pd.concat([results_df,wa],axis=1).to_csv("coach_alignment_results_may_chart.csv", index=False)